## Dataset

#### Dataset is a class where the data is organised such that when an index is given as a query, the corresponding element is returned.
#### Dataset is useful because it helps keep the preprocessing steps in one class (inherihed from the Dataset classs) and the training part in another class (inherited from the nn.Module) class

#### There are three commonly used functions:
#### i) `__init__`(self, <dataset>): It is used to define the dataset or use the dataset as a parameter and then organise the data for further processing
#### ii) `__len__`(self): It returns the size of the dataset
#### iii) `__getitem__`(self, i): It takes index i as the query and returns the ith element.

In [3]:
import torch
from torch.utils.data import Dataset


class SquareDataset(Dataset):
    def __init__(self):
        self.x = torch.tensor([_ for _ in range(1, 6)])
        self.y = self.x**2
    
    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        x_i = self.x[i]
        y_i = self.y[i]

        return (x_i, y_i)

In [4]:
dataset = SquareDataset()
print(len(dataset))
print(dataset[3])

5
(tensor(4), tensor(16))


#### In the above code, SquareDataset is a class which inherits Dataset class from the torch.utils.data package
#### Here, x = tensor([1, 2, 3, 4, 5]) and y = tensor([1, 4, 9, 16, 25])
#### dataset is an object. When it is created, the tensors x and y gets created. When we call len(dataset), it will return the length of the dataset, in this case 5. And to access the ith index (or the datapoint), dataset[i] is used and it gives the (x_i, y_i).

## DataLoader
#### DataLoader is another class from torch.utils.data
#### DataLoader takes the dataset and then splits it into batches and possibly shuffled and loaded in parallel.
# Some of the important attributes are:
## i) batch_size: It groups the data points into batches of the specified size

In [8]:
import torch
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False
)

for batch_idx, (x_b, y_b) in enumerate(loader):
    print()
    print("Batch: ", batch_idx)
    print("x_b: ", x_b)
    print("y_b", y_b)


Batch:  0
x_b:  tensor([1, 2])
y_b tensor([1, 4])

Batch:  1
x_b:  tensor([3, 4])
y_b tensor([ 9, 16])

Batch:  2
x_b:  tensor([5])
y_b tensor([25])


#### In the above example, we use the dataset defined in the Dataset part. There are 5 elements (n = 5). And the batch size = 2. So there will be 3 batches. The calculation is given below:
#### n = 5
#### batch_size = 2
#### No.of batches = math.ceil(n/batch_size) => math.ceil(5/2) = math.ceil(2.5) = 3

#### batch_size=2 means there are 2 data points in each batch.

## ii) shuffle: It generates random permutation of indices.

In [10]:
import torch
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

for batch_idx, (x_b, y_b) in enumerate(loader):
    print()
    print("Batch: ", batch_idx)
    print("x_b: ", x_b)
    print("y_b", y_b)


Batch:  0
x_b:  tensor([5, 1])
y_b tensor([25,  1])

Batch:  1
x_b:  tensor([2, 4])
y_b tensor([ 4, 16])

Batch:  2
x_b:  tensor([3])
y_b tensor([9])


## iii) num_workers: It performs parallel loading
#### Default value is 0. If num_workers>0, it spawns workers where the process is done in parallel.
#### To understand why num_workers is useful, let's first examine the high-level data flow from secondary storage to GPU VRAM during training.

![alt text](<Diagrams/High Level Data Flow of how num_workers = 2 parameter works in PyTorch .png>)

#### `num_workers = 2` means two worker processes are created on the CPU. Each worker fetches data by calling `dataset[index]`, which internally invokes the `Dataset.__getitem__(index)` method.

#### The `DataLoader` provides the indices to the workers. For example, if `batch_size = 2` and the selected indices are `[2, 5]`, one worker may execute `dataset[2]` while the other executes `dataset[5]`. Since both workers run in parallel, the two samples are loaded and preprocessed simultaneously. Once both samples are ready, the `DataLoader` combines them into a single batch using its `collate_fn`.